# user_configuration
Loads credentials from the Domo account store and exposes shared constants.
`%run` this at the top of every pipeline notebook.

In [ ]:
import os, json, time, base64, hashlib, secrets, tempfile, requests
from datetime import datetime, timezone, timedelta
from urllib.parse import urlencode, urljoin
from typing import Optional

# ── Domo ──────────────────────────────────────────────────────────────────────
try:
    import domojupyter as domo
except ImportError as e:
    raise RuntimeError("domojupyter is required in Domo Jupyter.") from e

# ── Credentials from Domo account store ───────────────────────────────────────
_creds_raw = domo.get_account_property_value("re_creds", "credentials")
_creds     = json.loads(_creds_raw)

SUBSCRIPTION_KEY = _creds["BB_SUBSCRIPTION_KEY"]
CLIENT_ID        = _creds["BB_CLIENT_ID"]
CLIENT_SECRET    = _creds["BB_CLIENT_SECRET"]
REDIRECT_URI     = _creds.get("BB_REDIRECT_URI", "https://api.domo.com/builder/oauth.html")

# ── Token bootstrap ────────────────────────────────────────────────────────────
# Priority: env var > Domo account store > empty string
# The actual latest refresh token lives in the "renxt_token_store" Domo dataset
# (written back by renxt_core after every successful refresh).
# The account store entry is only used as a manual bootstrap fallback.
PRESET_REFRESH_TOKEN   = os.getenv("BB_REFRESH_TOKEN") or _creds.get("BB_REFRESH_TOKEN") or ""
PRESERVE_REFRESH_TOKEN = True

# ── Blackbaud API constants ────────────────────────────────────────────────────
AUTH_URL  = "https://app.blackbaud.com/oauth/authorize"
TOKEN_URL = "https://oauth2.sky.blackbaud.com/token"
API_BASE  = "https://api.sky.blackbaud.com"

# ── Throttle / retry ───────────────────────────────────────────────────────────
CALLS_PER_SECOND        = 5
MAX_RETRIES             = 5
INITIAL_BACKOFF_SECONDS = 2
LIMIT                   = 500   # default API page size

# ── Token file cache (ephemeral — within a single run only) ───────────────────
WORK_DIR   = "./renxt_work"
TOKEN_PATH = os.path.join(WORK_DIR, ".sky_tokens.json")
os.makedirs(WORK_DIR, exist_ok=True)

print("✅ user_configuration loaded")
print(f"   CLIENT_ID set      : {bool(CLIENT_ID)}")
print(f"   CLIENT_SECRET set  : {bool(CLIENT_SECRET)}")
print(f"   REFRESH_TOKEN set  : {bool(PRESET_REFRESH_TOKEN)}")
